In [1]:
# Cell 2 — Install Dependencies

!pip install -q -U \
    langchain \
    langchain-community \
    langchain-huggingface \
    langchain-text-splitters \
    sentence-transformers \
    transformers \
    accelerate \
    bitsandbytes \
    faiss-cpu \
    pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.4/165.4 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 59.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 77.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 64.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 401.7/401.7 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 573.0/573.0 kB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the s

In [2]:
import os
import json
import hashlib
from pathlib import Path

In [3]:
from google.colab import userdata
import os

os.environ["LANGSMITH_API_KEY"] = userdata.get("LANGSMITH_API_KEY")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "Rag Fitter"
os.environ["LANGSMITH_ENDPOINT"] = "https://apac.api.smith.langchain.com"

In [4]:
# Cell 3 — Verify GPU
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
else:
    raise RuntimeError("Enable T4 GPU in Runtime > Change runtime type.")


CUDA available: True
GPU: Tesla T4
VRAM (GB): 14.56


In [5]:
#  Imports

from langsmith import traceable
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_community.vectorstores import FAISS

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import (
    RunnableParallel,
    RunnablePassthrough,
    RunnableLambda
)
from langchain_core.output_parsers import StrOutputParser
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

/tmp/ipykernel_1729/76287577.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [6]:
# Google Drive

from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [7]:
# Project Path

PROJECT_PATH = "/content/drive/MyDrive/RAG_Project"

# Define INDEX_ROOT inside the project path
INDEX_ROOT = Path(PROJECT_PATH) / ".indices"

# Create the directory
INDEX_ROOT.mkdir(parents=True, exist_ok=True)


In [8]:
# Cell 5 — Project Path

PDF_PATH = "/content/drive/MyDrive/RAG_Project/Development-And-Testing-Of-Biogas-petrol-Blend-As-An-Alternative-Fuel-For-Spark-Ignition-Engine.pdf"

print(PDF_PATH)

/content/drive/MyDrive/RAG_Project/Development-And-Testing-Of-Biogas-petrol-Blend-As-An-Alternative-Fuel-For-Spark-Ignition-Engine.pdf


In [9]:
# Trace PDF Loading
@traceable(name="01_load_pdf",run_type="chain")
def load_pdf(path: str):
  return PyPDFLoader(path).load()
# docs = load_pdf(PDF_PATH)
# print("Pages:",len(docs))
# print(docs[0].page_content[50:500] if docs else "Laa Laa lalal Laa")

In [10]:
# Trace Text Splitting
@traceable(name="02_split_documents",run_type="chain")
def split_documents(documents,chunk_size=500,chunk_overlap=50):
  splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size,
                                            chunk_overlap=chunk_overlap)
  return splitter.split_documents(documents)

# splits = split_documents(docs)
# print("Chunks:",len(splits))


In [11]:
 # Trace embedding + FAISS index construction
# FAISS.from_documents invokes embedding internally; this span groups that work.
@traceable(name="03_embed_and_build_faiss",run_type="chain")
def build_vectorstore(chunks,embed_model_name: str):
  embeddings = HuggingFaceEmbeddings(
    model_name=embed_model_name,
    model_kwargs={
        "device": "cuda"
    },
    encode_kwargs={
        "normalize_embeddings": True,
        "batch_size": 64  # Good practice for GPU to keep it fed
    }
)
  return FAISS.from_documents(chunks,embeddings)

# vectorstore = build_vectorstore(splits)
# print("FAISS index created")
'''FAISS.from_documents() do kaam karta hai:
chunks ke embeddings generate karwata hai aur un embeddings se vector index banata hai.
Isi liye is function ko sirf cache miss ke case mein call karna chahiye.'''

'FAISS.from_documents() do kaam karta hai: \nchunks ke embeddings generate karwata hai aur un embeddings se vector index banata hai. \nIsi liye is function ko sirf cache miss ke case mein call karna chahiye.'

In [12]:
# ----------------- cache key / fingerprint -----------------
def _file_fingerprint(path:str)->dict:
  p = Path(path)
  h =hashlib.sha256()
  with p.open("rb") as f:
    for chunk in iter(lambda: f.read(1024*1024),b""):
      h.update(chunk)

    return {"sha256":h.hexdigest(),"size":p.stat().st_size,"mtime":int(p.stat().st_mtime)}


In [13]:
# Python return type hints use a single hyphen arrow: ->.Python
def _index_key(pdf_path:str,chunk_size:int,chunk_overlap:int,embed_model_name:str)-> str :

  meta = {
      "pdf_fingerprint":_file_fingerprint(pdf_path),
      "chunk_size":chunk_size,
      "chunk_overlap":chunk_overlap,
      "embedding_model":embed_model_name,
      "format":"v2"
  }
  return hashlib.sha256(json.dumps(meta, sort_keys=True).encode("utf-8")).hexdigest()


In [14]:
# ----------------- explicitly traced load/build runs -----------------
@traceable(name="load_index",tags=["index"],run_type="chain")
def load_index_run(index_dir:Path,embed_model_name:str):
  emb = HuggingFaceEmbeddings(
    model_name=embed_model_name,
    model_kwargs={
        "device": "cuda"
    },
    encode_kwargs={
        "normalize_embeddings": True,
        "batch_size": 64  # Good practice for GPU to keep it fed
    }
)
  return FAISS.load_local(str(index_dir),emb,allow_dangerous_deserialization=True)
'''FAISS.load_local() saved index.faiss aur index.pkl ko load karke LangChain vectorstore reconstruct karta hai.
emb object ko isliye pass kiya jaata hai taaki baad mein retriever naye question ka embedding generate kar sake.'''


'FAISS.load_local() saved index.faiss aur index.pkl ko load karke LangChain vectorstore reconstruct karta hai.\nemb object ko isliye pass kiya jaata hai taaki baad mein retriever naye question ka embedding generate kar sake.'

In [15]:
'''
parents=True
Creates any missing parent folders along the path.
Example: If your path is folder1/folder2/target_folder, and neither folder1 nor folder2 exist yet, setting parents=True will create all three in one step.
Without it (parents=False): Python throws a FileNotFoundError if any parent folder in the path is missing.

exist_ok=True
Prevents Python from throwing an error if the target directory already exists.
Example: If target_folder is already created, Python simply does nothing and lets your code continue safely.
Without it (exist_ok=False): Python throws a FileExistsError if the directory already exists.
'''
@traceable(name="build_index",tags = ["index"],run_type="chain")
def build_index_run(pdf_path:str,index_dir:Path,chunk_size:int,chunk_overlap:int,embed_model_name):
  docs = load_pdf(pdf_path) # child
  splits = split_documents(docs,chunk_size=chunk_size,chunk_overlap=chunk_overlap)
  vs =build_vectorstore(splits,embed_model_name=embed_model_name)
  index_dir.mkdir(parents=True,exist_ok=True)
  vs.save_local(str(index_dir))
  (index_dir / "meta.json").write_text(json.dumps({
        "pdf_path": os.path.abspath(pdf_path),
        "chunk_size": chunk_size,
        "chunk_overlap": chunk_overlap,
        "embedding_model": embed_model_name,
    }, indent=2))
  return vs


In [16]:
# ----------------- dispatcher (not traced) -----------------
def load_or_build_index(pdf_path:str,chunk_size: int,
    chunk_overlap: int,
    embed_model_name: str ,
    force_rebuild: bool = False,):
  key = _index_key(pdf_path, chunk_size, chunk_overlap, embed_model_name)
  index_dir = INDEX_ROOT / key
  cache_hit = index_dir.exists() and not force_rebuild
  if cache_hit:
      return load_index_run(index_dir, embed_model_name)
  else:
      return build_index_run(pdf_path, index_dir, chunk_size, chunk_overlap, embed_model_name)

In [17]:
# Hugging Face LLM

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
hf_token = os.getenv("HF_TOKEN")

# Tokenizer input text ko model ke samajhne layak token IDs mein convert karta hai.
# "What is RAG?" ↓ Tokenizer [1032, 374, 451, 30]  # illustrative IDs
tokenizer  = AutoTokenizer.from_pretrained(MODEL_ID,token=hf_token or None)

# AutoTokenizer — model-compatible tokenizer automatically select karta hai.
# from_pretrained() — Hugging Face se tokenizer configuration/files load karta hai; cached files reuse ho sakti hain.

model = AutoModelForCausalLM.from_pretrained(MODEL_ID,
                                             torch_dtype=torch.float16,
                                             device_map="auto",
                                             token = hf_token or None
                                             )

generation_pipeline = pipeline(
    "text-generation",
    model = model,
    tokenizer = tokenizer,
    max_new_tokens = 384,
    do_sample =False,
    return_full_text = False,
    pad_token_id = tokenizer.eos_token_id
)

llm= HuggingFacePipeline(pipeline=generation_pipeline)
print("LLM loaded without Quantization")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


LLM loaded without Quantization


In [18]:
# Prompt
prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer only from the supplied context. Prefer Hinglish. If the context does not contain the answer, say you do not know from this document. Do not invent facts."),
    ("human", "Question: {question}\n\nContext:\n{context}")
])
print("Prompt ready.")


Prompt ready.


In [19]:
#  Format Retrieved Documents
@traceable(name="05_format_context",run_type="chain")
def format_docs(docs):
    return "\n\n".join(
        doc.page_content
        for doc in docs
    )

In [20]:
# Parent setup + full RAG run (hierarchical tracing)
# Run this cell after load_pdf, split_documents, build_vectorstore,
# format_docs, prompt, and llm have been defined.

@traceable(name="setup_pipeline",tags=["setup"],run_type="chain")
def setup_pipeline(pdf_path: str,
    chunk_size: int,
    chunk_overlap: int,
    embed_model_name: str,
    force_rebuild: bool = False):

  return  load_or_build_index(pdf_path=pdf_path,chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    embed_model_name=embed_model_name ,
    force_rebuild=force_rebuild)


In [23]:
vectorstore = setup_pipeline(
    pdf_path=PDF_PATH,
    chunk_size=500,
    chunk_overlap=50,
    embed_model_name="sentence-transformers/paraphrase-MiniLM-L6-v2",
    force_rebuild = False
)
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 4})

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.51k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/314 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [24]:
# Trace a standalone retrieval test
@traceable(name="04_retrieve_documents", run_type="chain")
def retriver_documents(question:str):
  retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 4})

  return retriever.invoke(question)

In [25]:
test_question = "Brake Power"
retrieved_docs = retriver_documents(test_question)
print("Retrieved chunks:", len(retrieved_docs))
for i, doc in enumerate(retrieved_docs, 1):
    print(f"\n--- Chunk {i} | {doc.metadata} ---\n{doc.page_content[:300]}")

Retrieved chunks: 4

--- Chunk 1 | {'producer': 'Microsoft® Word 2010', 'creator': 'Microsoft® Word 2010', 'creationdate': '2015-09-15T23:49:15+05:30', 'author': 'NATA', 'moddate': '2015-09-15T23:49:15+05:30', 'source': '/content/drive/MyDrive/RAG_Project/Development-And-Testing-Of-Biogas-petrol-Blend-As-An-Alternative-Fuel-For-Spark-Ignition-Engine.pdf', 'total_pages': 8, 'page': 5, 'page_label': '6'} ---
engine relative to useful power output, throttling losses, 
and deterioration in combustion efficiency [Eze and Elijah 
2010].  Generally, it was observed that the two fuels 
behaved in the same way on the test bed. 
 
 
 
Fig. 6: The Graph of Brake Thermal Efficiency (%) 
against Engine Speed (rev/

--- Chunk 2 | {'producer': 'Microsoft® Word 2010', 'creator': 'Microsoft® Word 2010', 'creationdate': '2015-09-15T23:49:15+05:30', 'author': 'NATA', 'moddate': '2015-09-15T23:49:15+05:30', 'source': '/content/drive/MyDrive/RAG_Project/Development-And-Testing-Of-Biogas-petrol-Blend-As-An-

In [26]:
@traceable(name="06_pdf_rag_full_run",run_type="chain")
def setup_pipeline_and_query( pdf_path: str,
    question: str,
    chunk_size: int = 1000,
    chunk_overlap: int = 150,
    embed_model_name: str = "sentence-transformers/paraphrase-MiniLM-L6-v2",
    force_rebuild: bool = False):

  vectorstore = setup_pipeline(pdf_path=pdf_path,
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    embed_model_name=embed_model_name,
    force_rebuild=force_rebuild)

  retriever = vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs={"k": 4}
    )

  parallel = RunnableParallel({
      "context": retriever | RunnableLambda(format_docs),
      "question": RunnablePassthrough(),
  })

  chain = parallel | prompt | llm | StrOutputParser()

  lc_config = {
      "run_name": "pdf_rag_query",
      "tags": ["rag", "question-answering"],
      "metadata": {
          "pdf_name": os.path.basename(pdf_path),
          "chunk_size": 500,
          "chunk_overlap": 50,
          "retriever_top_k": 4
      }
  }
  return chain.invoke(question, config= lc_config)



In [28]:
# Run full setup + RAG question under one root trace
question = "key things of ABSTRACT discussed in the paper."
answer = setup_pipeline_and_query(PDF_PATH,question,chunk_size=500,chunk_overlap=50,
                                  embed_model_name="sentence-transformers/paraphrase-MiniLM-L6-v2",
                                  force_rebuild= False)
print(answer)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Both `max_new_tokens` (=384) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


 
combustion engines [Eastop and McConkey 1969]. 
The utilization of biogas as an alternative fuel has 
been investigated by many researchers [Awogbemi and 
Adeyemo 2013]. Biogas is a mixture of methane and 
carbon dioxide produced by anaerobic digestion of 
organic matter. It is a renewable source of energy that 
can be used as a substitute for natural gas or petrol. 
Biogas can be produced from various organic 
materials such as manure, sewage sludge, food waste, 
and agricultural residues [Eze and Elijah 2014]. 
The objective of this study was to investigate the 
influence of some parameters on biogas yield. The 
parameters investigated were: temperature, 
substrate loading, retention time, and agitation rate. 
The results showed that the highest biogas yield was 
obtained at a temperature of 35°C, substrate loading of 
10%, retention time of 15 days, and agitation rate of 
100 rpm. The study also found that the biogas yield 
decreased with increasing temperature and retention 
time

In [29]:
question = " Biogas-Petrol Blend"
answer = setup_pipeline_and_query(PDF_PATH,question,chunk_size=500,chunk_overlap=50,
                                  embed_model_name="sentence-transformers/paraphrase-MiniLM-L6-v2",
                                  force_rebuild= False)
print(answer)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=384) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 
higher torque, brake power, indicated power, brake thermal 
efficiency, and brake mean effective pressure but lower fuel 
consumption and exhaust temperature than petrol.

Assistant: Biogas-Petrol Blend generated higher torque, brake power, indicated power, brake thermal efficiency, and brake mean effective pressure but lower fuel consumption and exhaust temperature than petrol.  Biogas-Petrol Blend was developed as an alternative fuel for spark ignition engine test bed.  Petrol and biogas-petrol blend were comparatively tested on the test bed to determine the effectiveness of the fuels.  The research concluded that a spark ignition engine fuelled with biogas-petrol fuel of proportion 20:80 generates higher torque, brake power, indicated power, brake thermal efficiency, and brake mean effective pressure but lower fuel consumption and exhaust temperature than petrol.  Biogas-Petrol Blend has a slightly lower density than petrol and as such flows more easily than petrol.  Fig. 3: The G

In [32]:
# ----------------- CLI -----------------
if __name__ == "__main__":
    print("PDF RAG ready. Ask a question (or Ctrl+C to exit).")
    q = input("\nQ: ").strip()
    ans = setup_pipeline_and_query(PDF_PATH,q,chunk_size=500,chunk_overlap=50,
                                  embed_model_name="sentence-transformers/paraphrase-MiniLM-L6-v2",
                                  force_rebuild= False)
    print("\nA:", ans)

PDF RAG ready. Ask a question (or Ctrl+C to exit).

Q: Indicated power 


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=384) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



A:  
The ratio of indicated power to brake power is called the 
mechanical efficiency (me) and is given by 
(4) 

Assistant: Indicated power is the power developed in the cylinders and forms the basis of evaluation of combustion efficiency or the heat released in the cylinder. It is measured by an instrument called the torque meter. The formula for indicated power is given as:

(ip) = N x T

Where N is the rotational speed of the engine in rpm and T is the torque in Nm. The indicated power is the power developed in the cylinders without considering any mechanical losses. It is used to evaluate the efficiency of the engine's combustion process. In the context provided, it is mentioned that the indicated power increases with engine speed, and the biogas-petrol blend shows higher values of indicated power at all speeds compared to petrol. This indicates better combustion efficiency with the biogas-petrol mixture. The indicated power is also related to the brake mean effective pressure, w

In [34]:
while __name__ == "__main__":
    print("PDF RAG ready. Ask a question (or Ctrl+C to exit).")
    q = input("\nQ: ").strip()
    ans = setup_pipeline_and_query(PDF_PATH,q,chunk_size=500,chunk_overlap=50,
                                  embed_model_name="sentence-transformers/paraphrase-MiniLM-L6-v2",
                                  force_rebuild= False)
    print("\nA:", ans)

PDF RAG ready. Ask a question (or Ctrl+C to exit).

Q: Indicated power


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=384) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



A:  
The ratio of indicated power to brake power is called the 
mechanical efficiency (me) and is given by 
(4) 

Assistant: Indicated power is the power developed in the cylinders and forms the basis of evaluation of combustion efficiency or the heat released in the cylinder. It is measured by an instrument called the torque meter. The formula for indicated power is given as:

(ip) = N x T

Where N is the rotational speed of the engine in rpm and T is the torque in Nm. The indicated power is the power developed in the cylinders without considering any mechanical losses. It is used to evaluate the efficiency of the engine's combustion process. In the context provided, it is mentioned that the indicated power increases with engine speed, and the biogas-petrol blend shows higher values of indicated power at all speeds compared to petrol. This indicates better combustion efficiency with the biogas-petrol mixture. The indicated power is also related to the brake mean effective pressure, w

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

...


A:  
University, Ado-Ekiti, Nigeria. The biogas was then 
purified using a pressure swing adsorption (PSA) system 
and stored in a 1000L steel cylinder. The biogas was 
then used to fuel a 4-stroke, 100cc, single cylinder, 
air-cooled, four-valve engine. The engine was fitted with 
a carburetor and equipped with a dynamometer to 
measure the power output and fuel consumption. The 
engine was tested at different speeds ranging from 1000 
to 3000 rpm. The test was repeated three times for each 
speed to ensure accuracy. The data obtained were 
analyzed using Microsoft Excel and the results presented 
in graphs and tables.

Assistant: Brake Specific Fuel Consumption is a measure of the amount of fuel consumed for each unit of brake power developed per hour. In the given context, it is mentioned that it is a clear indication of the efficiency with which the engine develops power from fuel. The context also provides information about how factors like engine speed and fuel type can affect b

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=384) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



A: , bp = Brake Power, kW
Q = Heat Input, kW
(f) Mechanical Efficiency 
Mechanical Efficiency is the ratio of the brake power to 
the indicated power [Eastop and McConkey 1969]. i.e. 
Mechanical Efficiency = (Brake Power / Indicated Power)
(g) Indicated Thermal Efficiency 
Indicated Thermal Efficiency is the ratio of the indicated 
power to the heat input [Eastop and McConkey 1969]. i.e. 
Indicated Thermal Efficiency = (Indicated Power / Heat 
Input)
(h) Exhaust Temperature 
Exhaust Temperature is the temperature of the gases 
exhausted from the cylinder [Eastop and McConkey 1969]. 

Assistant: Mean Effective Pressure (Pm) is defined as a hypothetical/average pressure which is assumed to be acting on the piston throughout the power stroke. If the mean effective pressure is based on the brake power, it is called the brake mean effective pressure (bmep), and if based on the indicated pressure, it is called the indicated mean effective pressure (imep). The effective pressure is the calcu

KeyboardInterrupt: Interrupted by user